# 💻 ParagAI: Lightweight Local CPU Training & Inference Guide

No GPU? No problem! **ParagAI** is engineered from first principles to be lightweight, modular, and fully functional on standard CPU hardware (Intel Core i3/i5/i7/i9, AMD Ryzen, or Apple Silicon).

---

### 🎯 Key Highlights for CPU Users:
- 🧠 **Minimal RAM Footprint:** Runs comfortably on 8GB–16GB system RAM.
- ⚡ **Multi-Core Threading:** Leverages all CPU cores via PyTorch MKL / OpenMP optimizations.
- 🔄 **Incremental Training:** Train 10–20 minutes daily without slowing down your computer.
- 🚀 **Fast CPU Inference in Ollama:** Ollama uses AVX2 SIMD instructions to generate text at 20–50+ tokens/sec on CPU!
- 🛡️ **Zero Data Loss:** Learn how to add new data day-by-day while keeping past knowledge intact.

## ⚙️ 1. Inspect System CPU & Allocate Processing Threads

In [ ]:
import os
import torch
import platform

num_cores = os.cpu_count() or 4
torch.set_num_threads(num_cores)

print(f"Operating System: {platform.system()} ({platform.machine()})")
print(f"CPU Logical Cores: {num_cores}")
print(f"PyTorch Threads:   {torch.get_num_threads()}")
print(f"PyTorch Version:   {torch.__version__}")
print(f"Target Device:     cpu")

## 🩺 2. Run Pre-Flight Diagnostics (`doctor.py`)
Inspects your environment and verifies that model tensor shapes and tokenizer roundtrips work smoothly on CPU.

In [ ]:
!python scripts/doctor.py

## 📚 3. Prepare Multi-Task SFT Data
Prepares your text corpus (stories, conversations, translations, and tools) with replay mixing.

In [ ]:
# Merge all raw text datasets into a single training file
!python scripts/prepare_data.py --no_tags

## 🔤 4. Verify or Train the Indic + English Tokenizer
Uses the pre-trained `checkpoints/tokenizer.json` or trains a new multilingual BPE tokenizer.

In [ ]:
!python scripts/train_multilingual_bpe.py \
    --input "data/raw/**/*.txt" \
    --vocab_size 4096 \
    --output checkpoints/tokenizer_indic.json \
    --sample_mb 5

## 🚀 5. Lightweight CPU Training
Runs an efficient CPU training session with small batch sizes and live telemetry progress.

In [ ]:
!python scripts/train.py \
    --data data/processed/combined_corpus.txt \
    --config configs/train_config.yaml \
    --model_config configs/model_small.yaml \
    --device cpu \
    --batch_size 16 \
    --max_iters 500

## 💬 6. Interactive Terminal Chat on CPU
Chat with the model directly in Python using standard greedy / nucleus sampling.

In [ ]:
import json
import os
import torch
from paragai.config import ModelConfig
from paragai.inference.generate import Generator
from paragai.model.transformer import ParagAIModel
from paragai.tokenizer.bpe_tokenizer import BPETokenizer
from paragai.tokenizer.char_tokenizer import CharTokenizer
from paragai.tokenizer.indic_bpe import IndicBPETokenizer

device = torch.device('cpu')
checkpoint = torch.load('checkpoints/best_model.pt', map_location=device)
model_config = ModelConfig(**checkpoint['model_config'])
model = ParagAIModel(model_config)
state_dict = {k: v for k, v in checkpoint['model_state_dict'].items() if not k.endswith('.attn.mask')}
model.load_state_dict(state_dict, strict=False)
model.eval()

tok_path = 'checkpoints/tokenizer.json' if os.path.exists('checkpoints/tokenizer.json') else 'checkpoints/tokenizer_indic.json'
with open(tok_path, 'r', encoding='utf-8') as f:
    tok_data = json.load(f)
if tok_data.get('tokenizer_class') == 'indic_bpe':
    tokenizer = IndicBPETokenizer.load(tok_path)
elif tok_data.get('type') == 'bpe':
    tokenizer = BPETokenizer.load(tok_path)
else:
    tokenizer = CharTokenizer.load(tok_path)

generator = Generator(model, tokenizer, device=device)

prompt = "<|im_start|>user\nHello! Tell me about ParagAI.<|im_end|>\n<|im_start|>assistant\n"
response = generator.generate(prompt, max_new_tokens=80, temperature=0.3, include_prompt=False)
print(response.split('<|im_end|>')[0].strip())

## 📊 7. Run Quantitative Benchmark Evaluation (`eval.py`)

In [ ]:
!python scripts/eval.py \
    --checkpoint checkpoints/best_model.pt \
    --tokenizer checkpoints/tokenizer.json \
    --tasks multiling,tools,story

## ⚡ 8. Export to GGUF and Run in Ollama (High-Speed CPU Serving)
Even on CPU, Ollama runs GGUF models at lightning speed using llama.cpp's AVX2 optimized CPU kernels.

In [ ]:
# 8.1 Export to SafeTensors
!python scripts/export_hf.py --checkpoint checkpoints/best_model.pt --output_dir checkpoints/export_hf --verify

# 8.2 Convert to GGUF with GPT-2 Unicode byte mapping
!python scripts/export_gguf.py --input_dir checkpoints/export_hf --tokenizer checkpoints/tokenizer.json --output checkpoints/paragai.gguf

# 8.3 Register into Ollama
!ollama create paragai -f deploy/Modelfile

# 8.4 Run instantly in terminal or query via CLI
!ollama run paragai:latest "Hello! Who are you?"